# 🕵️ SQL Murder Mystery — Investigación

**Team Challenge SQL · Parte I** — Alejandro Dietta

> *A crime has taken place and the detective needs your help. [...] You vaguely remember that the crime was a **murder** that occurred sometime on **Jan.15, 2018** and that it took place in **SQL City**.*

Juego creado por Joon Park y Cathy He ([mystery.knightlab.com](http://mystery.knightlab.com)). En este notebook se reproduce la investigación atacando la base de datos local `data/sql-murder-mystery.db` con las mismas queries usadas en la web.

## 0. Conexión a la base de datos

Se abre el fichero `.db` y se copia a una base de datos **en memoria**. Así, el `INSERT` final en la tabla `solution` no modifica el fichero versionado en el repositorio y el notebook se puede ejecutar tantas veces como se quiera.

La función `q()` ejecuta una query y devuelve el resultado como `DataFrame` para verlo en formato tabla.

In [2]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)  # ver descripciones y transcripciones completas

DB_PATH = Path("data/sql-murder-mystery.db")

disk = sqlite3.connect(DB_PATH)
conn = sqlite3.connect(":memory:")
disk.backup(conn)
disk.close()


def q(sql: str) -> pd.DataFrame:
    """Ejecuta una query SELECT y devuelve el resultado como DataFrame."""
    return pd.read_sql_query(sql, conn)


q("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,crime_scene_report
1,drivers_license
2,person
3,facebook_event_checkin
4,interview
5,get_fit_now_member
6,get_fit_now_check_in
7,income
8,solution


## 1. Informe de la escena del crimen

Sabemos tres cosas: fue un **asesinato** (`type = 'murder'`), el **15/01/2018** y en **SQL City**. En `crime_scene_report` la fecha se guarda como entero con formato `AAAAMMDD`, por eso va sin comillas.

In [3]:
q("""
SELECT *
FROM crime_scene_report
WHERE date = 20180115
    AND type = 'murder'
    AND city = 'SQL City';
""")

,date,type,description,city
0,20180115,murder,"Security footage shows that there were 2 witnesses. The first witness lives at the last house on ""Northwestern Dr"". The second witness, named Annabel, lives somewhere on ""Franklin Ave"".",SQL City


**Pista:** hay **2 testigos**:
1. El primero vive en **la última casa de "Northwestern Dr"**.
2. La segunda se llama **Annabel** y vive en **"Franklin Ave"**.

## 2. Identificar a los testigos

Los datos personales están en `person`. La dirección se divide en `address_number` y `address_street_name`.

**Testigo 1:** "la última casa" = el número más alto de la calle → ordenamos de mayor a menor y nos quedamos con la primera fila.

In [4]:
q("""
SELECT *
FROM person
WHERE address_street_name = 'Northwestern Dr'
ORDER BY address_number DESC
LIMIT 2;
""")

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
1,17729,Lasonya Wildey,439686,3824,Northwestern Dr,917817122


**Testigo 2:** solo conocemos el nombre de pila, así que usamos `LIKE 'Annabel%'` (empieza por *Annabel*).

In [5]:
q("""
SELECT *
FROM person
WHERE address_street_name = 'Franklin Ave'
    AND name LIKE 'Annabel%';
""")

,id,name,license_id,address_number,address_street_name,ssn
0,16371,Annabel Miller,490173,103,Franklin Ave,318771143


**Testigos identificados:**

| id | Nombre | Dirección |
|---|---|---|
| 14887 | Morty Schapiro | 4919 Northwestern Dr |
| 16371 | Annabel Miller | 103 Franklin Ave |

## 3. Declaraciones de los testigos

La tabla `interview` guarda las transcripciones por `person_id`. Hacemos `JOIN` con `person` para ver el nombre de quien declara.

In [6]:
q("""
SELECT witness.name, testimony.transcript
FROM interview AS testimony
JOIN person AS witness ON witness.id = testimony.person_id
WHERE testimony.person_id IN (14887, 16371);
""")

,name,transcript
0,Morty Schapiro,"I heard a gunshot and then saw a man run out. He had a ""Get Fit Now Gym"" bag. The membership number on the bag started with ""48Z"". Only gold members have those bags. The man got into a car with a plate that included ""H42W""."
1,Annabel Miller,"I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th."


**Pistas:**
- **Morty:** un **hombre** con bolsa de *Get Fit Now Gym*, nº de socio que **empieza por "48Z"** (solo socios **gold**), huyó en un coche con matrícula que **contiene "H42W"**.
- **Annabel:** reconoció al asesino de su gimnasio el **9 de enero** (2018).

## 4. Cruzar las pistas: gimnasio + check-in + carnet de conducir

Cada pista está en una tabla distinta, así que las encadenamos con `JOIN`:

| Pista | Tabla |
|---|---|
| Socio "48Z…", gold | `get_fit_now_member` |
| Fue al gimnasio el 09/01/2018 | `get_fit_now_check_in` |
| Une socio con carnet | `person` (`person_id` → `license_id`) |
| Matrícula con "H42W", hombre | `drivers_license` |

`LIKE '48Z%'` = *empieza por*; `LIKE '%H42W%'` = *contiene*.

In [7]:
q("""
SELECT member.id AS membership_id, member.name, member.membership_status,
    checkin.check_in_date, license.plate_number, license.gender,
    suspect.id AS person_id
FROM get_fit_now_member AS member
JOIN get_fit_now_check_in AS checkin ON checkin.membership_id = member.id
JOIN person AS suspect ON suspect.id = member.person_id
JOIN drivers_license AS license ON license.id = suspect.license_id
WHERE member.id LIKE '48Z%'
    AND member.membership_status = 'gold'
    AND checkin.check_in_date = 20180109
    AND license.plate_number LIKE '%H42W%'
    AND license.gender = 'male';
""")

,membership_id,name,membership_status,check_in_date,plate_number,gender,person_id
0,48Z55,Jeremy Bowers,gold,20180109,0H42W2,male,67318


**Sospechoso:** solo una persona cumple todas las pistas → **Jeremy Bowers** (`person_id` 67318).

## 5. Comprobar la solución

El juego valida la respuesta insertando el nombre en la tabla `solution`: un *trigger* de la base de datos sustituye la fila por un mensaje indicando si es correcto. Como trabajamos sobre la copia en memoria, el fichero `.db` no se modifica.

In [8]:
conn.execute("INSERT INTO solution VALUES (1, 'Jeremy Bowers')")
conn.commit()
q("SELECT value FROM solution")

,value
0,"Congrats, you found the murderer! But wait, there's more... If you think you're up for a challenge, try querying the interview transcript of the murderer to find the real villain behind this crime. If you feel especially confident in your SQL skills, try to complete this final step with no more than 2 queries. Use this same INSERT statement with your new suspect to check your answer."


## 6. ¿Quién está detrás del crimen?

El mensaje de la solución anima a leer la declaración del asesino para encontrar a quien lo contrató, con el reto de hacerlo en **2 queries como máximo**.

**Query 1:** declaración de Jeremy Bowers (`person_id` 67318).

In [ ]:
q("""
SELECT transcript
FROM interview
WHERE person_id = 67318;
""")

**Pistas sobre quien le contrató:**

| Pista | Tabla | Condición |
|---|---|---|
| Mujer | `drivers_license` | `gender = 'female'` |
| Entre 65" y 67" | `drivers_license` | `height BETWEEN 65 AND 67` |
| Pelo rojo | `drivers_license` | `hair_color = 'red'` |
| Tesla Model S | `drivers_license` | `car_make`, `car_model` |
| 3 veces al *SQL Symphony Concert* en dic. 2017 | `facebook_event_checkin` | `GROUP BY` + `HAVING COUNT(*) = 3` |
| "Con mucho dinero" | `income` | `annual_income` (solo para confirmar) |

**Query 2:**
- `INNER JOIN` con `person` y `facebook_event_checkin` porque son **requisitos** de la búsqueda.
- `LEFT JOIN` con `income` porque es **información de apoyo**: no todas las personas tienen ingresos registrados y no queremos perder a la sospechosa por ello.
- `HAVING` filtra después de agrupar, que es lo que permite contar los check-ins por persona. Todas las columnas no agregadas del `SELECT` van en el `GROUP BY` (SQL estándar, obligatorio en motores como BigQuery).

In [ ]:
q("""
SELECT villain.id AS person_id, villain.name, license.height, license.hair_color,
    license.car_make, license.car_model, earnings.annual_income,
    COUNT(*) AS concert_checkins
FROM drivers_license AS license
JOIN person AS villain ON villain.license_id = license.id
JOIN facebook_event_checkin AS checkin ON checkin.person_id = villain.id
LEFT JOIN income AS earnings ON earnings.ssn = villain.ssn
WHERE license.gender = 'female'
    AND license.height BETWEEN 65 AND 67
    AND license.hair_color = 'red'
    AND license.car_make = 'Tesla'
    AND license.car_model = 'Model S'
    AND checkin.event_name = 'SQL Symphony Concert'
    AND checkin.date BETWEEN 20171201 AND 20171231
GROUP BY villain.id, villain.name, license.height, license.hair_color,
    license.car_make, license.car_model, earnings.annual_income
HAVING COUNT(*) = 3;
""")

**Mente maestra:** **Miranda Priestly** (`person_id` 99716) — 66", pelo rojo, Tesla Model S, 3 check-ins en el concierto e ingresos de 310.000 $/año.

Comprobamos la respuesta con el mismo `INSERT` que antes:

In [ ]:
conn.execute("INSERT INTO solution VALUES (1, 'Miranda Priestly')")
conn.commit()
q("SELECT value FROM solution")

## ✅ Conclusión

| Paso | Tablas | Resultado |
|---|---|---|
| 1. Informe del crimen | `crime_scene_report` | 2 testigos: última casa de Northwestern Dr y Annabel en Franklin Ave |
| 2. Testigos | `person` | Morty Schapiro (14887) y Annabel Miller (16371) |
| 3. Declaraciones | `interview`, `person` | Socio gold "48Z…", matrícula con "H42W", en el gimnasio el 09/01 |
| 4. Sospechoso | `get_fit_now_member`, `get_fit_now_check_in`, `person`, `drivers_license` | **Jeremy Bowers** (asesino) |
| 6. Mente maestra | `interview`, `drivers_license`, `person`, `facebook_event_checkin`, `income` | **Miranda Priestly** |

**Jeremy Bowers** cometió el asesinato, contratado por **Miranda Priestly**. La mente maestra se identificó en 2 queries, como proponía el reto.

**Conceptos SQL aplicados:** filtros con `WHERE`, `LIKE` y `BETWEEN`; `ORDER BY` + `LIMIT`; `INNER JOIN` y `LEFT JOIN` encadenados; agregación con `GROUP BY`, `COUNT` y `HAVING`; `INSERT` para validar la solución (sobre una copia en memoria de la base de datos).

In [ ]:
conn.close()